### Bài 1 ###

* Prelude
    - Ta có công thức của hàm softmax và cross-entropy lần lượt là: $$P(c|x)=\frac{exp(o_c)}{\sum_{c'=1}^{C}exp(o_{c'})}, c = 1,2,..,C$$ và $$L(o,y)=-\sum_{c=1}^{C}I(y=c)log(P(c|x)$$Kết hợp hai công thức trên, ta có $$L(o,y) = -\sum_{c=1}^{C}I(y=c)log\frac{exp(o_c)}{\sum_{c'=1}^{C}exp(o_{c'})}$$ và thay thế, ta có$$L(o, y) = -\sum_{c=1}^C I(y=c) \left[ o_c - \log\left(\sum_{c'=1}^C \exp(o_{c'})\right) \right] (1)$$ khi đó, nếu y là một nhãn one-hot thì kết quả cuối cùng là $$L(o, y) = -y + \log\left(\sum_{c'=1}^C \exp(o_{c'})\right) (2)$$

* Câu 1
    - Đạo hàm hàm loss (1) theo đầu ra $o$:
        - Nếu là lớp đúng thì đạo hàm là: $$-1 + \frac{1}{\sum \exp(o_j)} \times \frac{\partial (\sum \exp(o_j))}{\partial o_{target}} = \frac{\exp(o)}{\sum \exp(o_j)}-1 = o-1$$
        - Tương tự, nếu là lớp sai thì đạo hàm là: $$o-0$$
    - Vector hóa hai công thức trên, với y là nhãn one-hot, ta có: $$\frac{\partial L}{\partial o} = o - y$$
    - Với lớp a trong mạng, ta có đầu ra của lớp là $o_a$, gradient được lan truyền ngược từ lớp a + 1 là $\frac{\partial L}{\partial o_{a+1}}$, hàm kích hoạt là $\sigma$, ta có đạo hàm đầu ra theo lớp là $$\frac{\partial L}{\partial o_a} = W_{a+1} \times \sigma'(z_{a+1}) \times \frac{\partial L}{\partial o_{a+1}}$$ cho đến khi $o_{a+1}$ là đầu ra lớp cuối cùng
* Câu 2
    - Với đầu ra thứ a của lớp $l$, ta có: $$\frac{\partial L}{\partial o_{l,a}} = W_{l,a+1} \times \sum \sigma'(o_{a+1}) \times \frac{\partial L}{\partial o_{l,a+1}}$$
* Câu 3
    - Ta có $$\frac{\partial L}{\partial W_l} = o_{l-1} \times \sigma'(z_l) \times \frac{\partial L}{\partial o_l}$$ và $$\frac{\partial L}{\partial b_l} =\sigma'(z_l) \times \frac{\partial L}{\partial o_l}$$
* Câu 4
    - Với SGD, batch size = 1, learning rate = $\eta$, ta có $$W_l = W_l - \eta\frac{\partial L}{\partial W_l}$$ và $$b_l = b_l - \eta\frac{\partial L}{\partial b_l}$$

### Bài 2

In [67]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score
import pandas as pd
import numpy as np
import random

digits = load_digits()

X_train, X_test, y_train, y_test = train_test_split(digits.data,digits.target,test_size=0.4,random_state=40)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=40
)

In [68]:
#Batch for Neural network
BATCH_SIZE = 128
num_batch = len(X_train) // BATCH_SIZE
batched_data = []
for i in range(num_batch):
    start_idx = i * BATCH_SIZE
    end_idx = (i + 1) * BATCH_SIZE
    X_batch = X_train[start_idx:end_idx]
    y_batch = y_train[start_idx:end_idx]
    batched_data.append((X_batch, y_batch))
test_batched_data = []
num_test_batch = len(X_test) // BATCH_SIZE
for i in range(num_test_batch):
    start_idx = i * BATCH_SIZE
    end_idx = (i + 1) * BATCH_SIZE
    X_batch = X_test[start_idx:end_idx]
    y_batch = y_test[start_idx:end_idx]
    test_batched_data.append((X_batch, y_batch))
val_batch_data = []
num_val_batch = len(X_val) // BATCH_SIZE
for i in range(num_val_batch):
    start_idx = i * BATCH_SIZE
    end_idx = (i + 1) * BATCH_SIZE
    X_batch = X_val[start_idx:end_idx]
    y_batch = y_val[start_idx:end_idx]
    val_batch_data.append((X_batch, y_batch))

In [69]:
def relu(x):
    return np.maximum(0, x)
def derivative_relu(x):
    return (x > 0).astype(float)
def softmax(x):
    max_x = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - max_x)
    sum_exp_x = np.sum(exp_x, axis=1, keepdims=True)
    return exp_x / sum_exp_x
def cross_entropy_error(y_pred, y_true):
    num_samples = y_pred.shape[0]
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1.0 - epsilon)
    loss_prob = -np.sum(y_true * np.log(y_pred))
    return loss_prob / num_samples


class Layer:
    def __init__(self, input_size, output_size):
        std = np.sqrt(2.0 / input_size)
        self.W = np.random.randn(input_size,output_size) * std
        self.b = np.zeros(output_size)
        self.output = 0
        self.input = 0
    def forward(self, in_layer):
        self.input = in_layer
        self.output = np.dot(self.input, self.W) + self.b
        return relu(self.output)
    def backward(self, gradient_layer, learning_rate, max_norm = 5):
        gradient_layer = gradient_layer * derivative_relu(self.output)
        dW = np.dot(self.input.T, gradient_layer) / self.input.shape[0]
        db = np.mean(gradient_layer, axis=0)
        norm_W = np.linalg.norm(dW)
        if norm_W > max_norm:
            dW = dW * (max_norm / norm_W)
        norm_b = np.linalg.norm(db)
        if norm_b > max_norm:
            db = db * (max_norm / norm_b)
        self.W -= dW * learning_rate
        self.b -= db * learning_rate
        return np.dot(gradient_layer, self.W.T)
class NeuralNetwork:
    def __init__(self, input_Size,hidden_Size, output_Size):
        self.layer = []
        self.HiddenNumber = len(hidden_Size)
        self.layerIn = Layer(input_size=input_Size, output_size=hidden_Size[0])
        for layer_num in range(self.HiddenNumber - 1):
            self.layer.append(Layer(input_size=hidden_Size[layer_num], output_size=hidden_Size[layer_num + 1]))
        self.layer.append(Layer(input_size=hidden_Size[-1], output_size=output_Size))
        self.output = 0
    def predict(self, X):
        in_hidden = self.layerIn.forward(X)
        for layer in self.layer:
            in_hidden = layer.forward(in_hidden)
        return softmax(self.layer[-1].output)
    def backprop(self, loss_gradient, learning_rate, max_norm = 5):
        dW = np.dot(self.layer[-1].input.T, loss_gradient) / self.layer[-1].input.shape[0]
        db = np.mean(loss_gradient, axis=0)
        norm_W = np.linalg.norm(dW)
        if norm_W > max_norm:
            dW = dW * (max_norm / norm_W)
        norm_b = np.linalg.norm(db)
        if norm_b > max_norm:
            db = db * (max_norm / norm_b)

        self.layer[-1].W -= dW * learning_rate
        self.layer[-1].b -= db * learning_rate

        loss_gradient = np.dot(loss_gradient, self.layer[-1].W.T)

        for layer in range(self.HiddenNumber - 2, -1, -1):
            loss_gradient = self.layer[layer].backward(loss_gradient, learning_rate)

        self.layerIn.backward(loss_gradient, learning_rate=learning_rate)

In [70]:
knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
dtree = DecisionTreeClassifier(criterion='gini', max_depth=4)
logit_classifier = LogisticRegression(max_iter=1000)
nn_model = NeuralNetwork(64,[32,16],10)

In [71]:
lr = 0.01
epochs = 1000
list_loss = []
def to_one_hot(labels, num_classes):
    one_hot = np.zeros((labels.size, num_classes))
    one_hot[np.arange(labels.size), labels.flatten()] = 1
    return one_hot
for epoch in range(epochs):
    random.shuffle(batched_data)
    for X,y in batched_data:
        X = np.array(X)
        y = np.array(y)
        pred = nn_model.predict(X)
        loss = cross_entropy_error(pred, to_one_hot(y, num_classes=10))
        grad_loss = pred - to_one_hot(y,num_classes=10)
        nn_model.backprop(grad_loss, learning_rate=lr)

knn.fit(X_train, y_train)
dtree.fit(X_train, y_train)
logit_classifier.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [72]:
#valdiation check
y_pred_knn = knn.predict(X_val)
accuracy_knn = accuracy_score(y_val, y_pred_knn)
accuracy_dtree = accuracy_score(y_val, dtree.predict(X_val))
accuracy_logit_classifier = accuracy_score(y_val, logit_classifier.predict(X_val))
accuracy_nn = accuracy_score(y_val, np.argmax(nn_model.predict(X_val),axis=1))
print(accuracy_knn, accuracy_dtree, accuracy_logit_classifier, accuracy_nn)

0.9814814814814815 0.5777777777777777 0.9629629629629629 0.9629629629629629


In [73]:
y_pred_knn = knn.predict(X_test)
y_pred_dtree = dtree.predict(X_test)
y_pred_logit = logit_classifier.predict(X_test)
y_pred_nn = np.argmax(nn_model.predict(X_test), axis=1)
models = {
    "KNN": y_pred_knn,
    "Decision Tree": y_pred_dtree,
    "Logistic Regression": y_pred_logit,
    "Neural Network": y_pred_nn,
}
results_list = []
for model_name, y_pred in models.items():
    accuracy = accuracy_score(y_test, y_pred)
    class_recall = recall_score(y_test, y_pred, average=None)
    row = {
        "Mô hình": model_name,
        "Accuracy (Test)": f"{accuracy:.4f}",
        **{f"Lớp {i}": f"{class_recall[i]:.4f}" for i in range(10)}
    }
    results_list.append(row)
df_results = pd.DataFrame(results_list)
column_order = ["Mô hình", "Accuracy (Test)"] + [f"Lớp {i}" for i in range(10)]
df_results = df_results[column_order]
df_results

,Mô hình,Accuracy (Test),Lớp 0,Lớp 1,Lớp 2,Lớp 3,Lớp 4,Lớp 5,Lớp 6,Lớp 7,Lớp 8,Lớp 9
0,KNN,0.9819,1.0000,1.0000,0.9857,0.9855,0.9859,0.9833,1.0000,1.0000,0.9231,0.9605
1,Decision Tree,0.5327,0.9718,0.0000,0.1571,0.7971,0.7324,0.8833,0.8750,0.0563,0.7436,0.2368
2,Logistic Regression,0.9430,1.0000,0.9136,0.9857,0.9710,0.8732,0.9667,0.9861,0.9859,0.8590,0.9079
3,Neural Network,0.9291,0.9859,0.8148,0.9286,0.9565,0.9155,0.9500,0.9861,0.9577,0.8974,0.9211
